<a href="https://colab.research.google.com/github/aungKhantPaing/colab/blob/main/XML%20for%20Contracts/XPath%20Hand-On%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/XML%20for%20Contracts/XPath%20Hand-On%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 Setting Up the XPath Environment

**XPath** (XML Path Language) is a query language for selecting nodes — elements, attributes, and text — from an XML document. In this lab you run XPath with the **`%%xpath` magic**, the XML cousin of the `%%mongodb` magic from Chapter 6. It evaluates expressions with `xmllint`, so you write XPath instead of boilerplate parsing code.

> **About the `%%xpath` magic.** `cellspell` registers two helpers:
> - `%xpath file.xml` (line magic) — checks that `file.xml` is **well-formed**; add `--xsd schema.xsd` or `--dtd file.dtd` to **validate** against a schema.
> - `%%xpath file.xml` (cell magic, must be the **first line** of a cell) — runs the XPath expression on the **next line(s)** against `file.xml` and prints the result.
>
> *This lab uses `%%xpath` for **extraction** only; validating XML against an **XSD** or **DTD** (`%xpath --xsd ...` / `--dtd ...`) is covered in `07-lab-xml-contracts`.*

**Setup — install `xmllint` and `cellspell`, then load the `%%xpath` magic:**

In [11]:
# Step 1 — install xmllint (libxml2-utils) and cellspell, then load the %%xpath magic
!apt-get update -qq && apt-get install -y -qq libxml2-utils > /dev/null 2>&1
!pip install "cellspell[xpath] @ git+https://github.com/sreent/jupyter-query-magics.git" -q
%load_ext cellspell.xpath

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
The cellspell.xpath extension is already loaded. To reload it, use:
  %reload_ext cellspell.xpath


**Verify `xmllint` is available:**

In [12]:
!xmllint --version 2>&1 | head -1
# Expected: xmllint using libxml version ...

xmllint: using libxml version 20913


# 2 Sample XML Data

We will query a small `<library>` of books. Create the file directly from this cell with `%%writefile` — no upload needed:

In [13]:
%%writefile library.xml
<library>
    <book id="1">
        <title>Python Programming</title>
        <author>John Doe</author>
        <year>2020</year>
        <price>29.99</price>
    </book>
    <book id="2">
        <title>Learning XPath</title>
        <author>Jane Smith</author>
        <year>2019</year>
        <price>19.99</price>
    </book>
    <book id="3">
        <title>Data Science Handbook</title>
        <author>Emily Davis</author>
        <year>2018</year>
        <price>39.99</price>
    </book>
</library>

Overwriting library.xml


**Check that the document is well-formed (line magic — note the single `%`):**

In [14]:
%xpath library.xml

✓ Valid


> **Expected:** the magic reports the document is well-formed. Well-formedness checks XML *syntax* (matching tags, proper nesting) — it does not check any schema.

# 3 Your First XPath Query

The `%%xpath` magic takes the **file on the first line** and the **XPath expression on the next line**. Here is a complete, worked example — run it to list every book title. `//book` selects every `book` element at any depth, `/title` steps into its `title` child, and `/text()` extracts the text:

In [15]:
%%xpath library.xml
//book/title/text()

Python Programming
Learning XPath
Data Science Handbook



> **Expected:** three titles — *Python Programming*, *Learning XPath*, *Data Science Handbook*.

From here on, **you write the XPath** — type it on the line below `%%xpath library.xml`, run the cell, then reveal the solution to check.

# 4 Basic Paths

A path walks down the tree one step at a time, and there are two ways to start:

- **Absolute path** — a leading `/` starts at the root, and each `/step` moves down **exactly one level**. The full child path to a title is `/library/book/title`.
- **Descendant shortcut** — `//` means *"at any depth"*, so `//book` finds every `book` wherever it sits. When an element name is unambiguous, `//` is shorter; the absolute path is more explicit about structure.

Each step has up to three parts: an **axis** (direction — child, descendant `//`, parent `..`), a **node test** (which nodes — a name like `book`, or the wildcard `*`), and an optional **predicate** (`[...]` filter). You'll use all three below.

**a. Write the *absolute* path from the root to every book title.** (Start at `/library`.)

In [16]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** the same three titles as the demo — *Python Programming*, *Learning XPath*, *Data Science Handbook*. This is `//book/title/text()` spelled out in full; `//` is the shortcut when the path is unambiguous.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
/library/book/title/text()
```

</details>

**b. Extract the `author` of the *first* book.**

In [17]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** *John Doe* (the `[1]` predicate selects the first `book`).

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book[1]/author/text()
```

</details>

**c. Extract every book's `price`.**

In [18]:
%%xpath library.xml
//book/


Error: No XPath expression provided.


> **Expected:** three prices — `29.99`, `19.99`, `39.99`.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book/price/text()
```

</details>

# 5 Predicates — Filtering with `[...]`

A predicate in square brackets keeps only the nodes that satisfy a condition. Conditions can compare child values (`year > 2018`) or attributes (`@id`).

**a. Titles of books published *after* 2018.**

In [19]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** *Python Programming* (2020) and *Learning XPath* (2019). The 2018 book is excluded.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book[year > 2018]/title/text()
```

</details>

**b. Titles of books that cost *more than* \$20.**

In [20]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** *Python Programming* (29.99) and *Data Science Handbook* (39.99). *Learning XPath* (19.99) is excluded.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book[price > 20]/title/text()
```

</details>

**c. The title of the book whose `id` attribute is `2`.** (Attributes are matched with `@`.)

In [31]:
%%xpath library.xml
//book[@id=2]/title


<title>Learning XPath</title>


> **Expected:** *Learning XPath*.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book[@id="2"]/title/text()
```

</details>

**d. Titles of books whose *author's name starts with* `J`.** Predicates can call string functions like `starts-with()` and `contains()`.

In [45]:
%%xpath library.xml
//book[starts-with(@author, 'J')]/title/text()


XPath error: XPath set is empty


> **Expected:** *Python Programming* (John Doe) and *Learning XPath* (Jane Smith).

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book[starts-with(author, "J")]/title/text()
```

</details>

# 6 Navigating — `//`, `..`, and `*`

Three more building blocks: `//` is the **descendant** shortcut (match at any depth), `..` is the **parent** axis (step **up** one level), and `*` is a **wildcard node test** (match *any* element name, not a specific one).

**a. Every `author` in the document, regardless of where it sits.**

In [46]:
%%xpath library.xml
//author/text()

John Doe
Jane Smith
Emily Davis



> **Expected:** *John Doe*, *Jane Smith*, *Emily Davis*.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//author/text()
```

</details>

**b. The `price` of the book titled "Python Programming"** — find the matching `title`, step up to its `book` parent with `..`, then down to `price`.

In [48]:
%%xpath library.xml
//title[text()="Python Programming"]/../price/text()


29.99



> **Expected:** `29.99`.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//title[text()="Python Programming"]/../price/text()
```

</details>

**c. Every *child element* of the first book**, using the wildcard `*`.

In [25]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** the four children of book 1 — `<title>`, `<author>`, `<year>`, `<price>`.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book[1]/*
```

</details>

# 7 Attributes and Functions

XPath has built-in functions for positions (`last()`, `position()`), truth tests (`boolean()`), and aggregation (`count()`).

**a. The `id` attribute of every book.** (Select attributes with `@`.)

In [26]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** the three ids — `1`, `2`, `3`.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book/@id
```

</details>

**b. The title of the *last* book**, using `last()`.

In [27]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** *Data Science Handbook*.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book[last()]/title/text()
```

</details>

**c. The titles of the *first two* books**, using `position()`.

In [28]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** *Python Programming* and *Learning XPath*.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
//book[position() <= 2]/title/text()
```

</details>

**d. Is there *any* book published in 2020?** Use `boolean(...)`.

In [29]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** `true` — *Python Programming* was published in 2020.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
boolean(//book[year=2020])
```

</details>

**e. How many books are in the library?** Use `count(...)`.

In [30]:
%%xpath library.xml


Error: No XPath expression provided.


> **Expected:** `3`.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```
%%xpath library.xml
count(//book)
```

</details>

# 8 Wrap-up

You queried an XML document end-to-end with the `%%xpath` magic — **absolute and descendant paths** (`/library/book` vs `//book`), **predicates** that filter on numbers, attributes, and string functions like `starts-with()`, the **`//`, `..`, and `*`** building blocks, and **attribute** (`@`) and **function** (`last()`, `position()`, `boolean()`, `count()`) expressions — with no parsing boilerplate. Every step you wrote was some combination of an **axis**, a **node test**, and a **predicate**.

Next, `07-lab-xml-contracts.ipynb` applies the same `%%xpath` magic to a real **e-invoicing** contract: validating XML against an **XSD**, comparing XSD with **DTD**, and extracting fields from a namespaced invoice.